In [ ]:
import pandas as pd
import numpy as np

# visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# evaluation
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
df = pd.read_csv("HRDataset_v14_enriched_augmented_1000_clean.csv")

df.head()

In [ ]:
cols_to_drop = [
    "Employee_Name",
    "EmpID",
    "ManagerName",
    "ManagerID",
    "Zip",
    "DOB",
    "DateofHire",
    "DateofTermination",
    "LastPerformanceReview_Date",
    "Feedback_RH"
]

df = df.drop(columns=cols_to_drop, errors="ignore")

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
for col in df.select_dtypes(include="object"):
    df[col] = df[col].fillna("Unknown")

for col in df.select_dtypes(include=np.number):
    df[col] = df[col].fillna(df[col].median())

In [ ]:
df["Termd"].value_counts()

In [ ]:
X = df.drop("Termd", axis=1)
y = df["Termd"]

In [ ]:
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

In [ ]:
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier()
}

In [ ]:
results = {}

for name, model in models.items():
    
    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )
    
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    
    results[name] = acc
    
    print("------", name, "------")
    print("Accuracy:", acc)
    print(classification_report(y_test, y_pred))

In [ ]:
results_df = pd.DataFrame(results.items(), columns=["Model", "Accuracy"])
results_df = results_df.sort_values(by="Accuracy", ascending=False)

results_df

In [ ]:
sns.barplot(data=results_df, x="Accuracy", y="Model")

plt.title("Comparaison des modèles de prédiction des démissions")
plt.show()

In [ ]:
rf_pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier())
    ]
)

rf_pipe.fit(X_train, y_train)